In [253]:
import pandas as pd 
import numpy as np 
from rapidfuzz.fuzz import token_sort_ratio, token_set_ratio
from rapidfuzz.distance import Levenshtein
import rapidfuzz

In [254]:

pd.set_option('display.max_colwidth', None)


In [255]:
df = pd.read_csv('nuforc_str.csv')
df.head()

,Sighting,Occurred,Location,Shape,Duration,No of observers,Reported,Posted,Characteristics,Summary,Text,Location details,Explanation
0,114864,2014-09-21 13:00:00 Local,"Huntsville, TX, USA",Rectangle,several seconds,1.0,2014-10-23 11:11:17 Pacific,2014-11-06 00:00:00,"Lights on object, Aura or haze around object",Rectangle shaped UFO observed traveling at an extremely high rate of speed and sending out bright white propulsion as it traveled.,I observed a rectangle shaped UFO moving at a very high rate of speed and sending out bright white propulsion as it traveled. The UFO seemed to be larger than an aircraft as I noticed that as it flew under a cloud. \nMy camera was aimed at 90 degrees upward..straight up in the sky when the UFO was recorded. The UFO itself was dark in color.,NaN,NaN
1,126755,2015-12-18 13:00:00 Local,"Sonoma, CA, USA",Sphere,2 minutes,1.0,2016-04-07 13:23:15 Pacific,2016-04-15 00:00:00,NaN,"Round, metallic-looking sphere over my house","It was a bright, clear day in December 2015, about 60 degrees and sunny, and I went out to get some firewood.\nI happened to look up and there was a round (spherical), bronze-colored craft sitting stationary over my house.\nIt had a metallic looking appearance and on the side that I was looking at it looked to me like there were part concave or convex areas to it. I was looking at it facing north, looking up over my house.\nThere were no propellers, so it could not have been a drone, and no sound. It was approximately 300 ft. above my house and if I were to estimate width, I would say maybe 20 to 30 feet wide.\nI was the only witness and I watched it for about 1 to 2 minutes. Here is another additional part to my story.\nI have followed the UFO media reports for quite a few years and always thought that it would be great to see one, to validate for myself that they exist. I would get my camera, get a witness, document everything, etc.\nAfter watching the craft, I continued on out to the wood pile and got the logs for my fireplace, and forgot about the entire thing… I remembered about 3 ½ weeks later, into January of this year , about witnessing the craft.\nI am totally perplexed about that part of my story.\nI am a 62 year old woman with grandkids, have run a business for many years, and have lived here for 28 years. There would be no reason for me to fabricate this event. As to who was piloting it - ??",NaN,NaN
2,106946,2014-02-05 21:00:00 Local,"Hershey, PA, USA",Light,10 seconds,1.0,2014-02-05 19:04:05 Pacific,2014-02-07 00:00:00,NaN,Bright flash of light. solid white object in the sky. disappeared after a couple seconds. Seen while driving .,"I was driving on the highway and there was a huge flash of light. The light lit up the entire town, possibly more. \nAs I looked around for a split second I noticed a very bright ball of light in the sky ahead of. \nThis light was almost tear dropped. It was moving downwards at a high rate. \nAfter a couple seconds it disappeared. \nMy dad inside the house with closed blinds saw this flash of light. \nThe house is about 2-3 minutes from my location at the time, in a neighborhood with houses all around.\nWitness confirms a nighttime sighting. We have amended the time above. PD",NaN,NaN
3,161419,2019-05-16 12:00:00 Local,"Brownsville, TX, USA",Oval,NaN,1.0,2021-01-02 20:45:43 Pacific,2021-01-19 00:00:00,"Aircraft nearby, Animals reacted",NaN,NaN,NaN,NaN
4,18735,2001-07-29 23:59:00 Local,"Tucson, AZ, USA",Unknown,?,2.0,2001-07-30 00:00:00 Pacific,2001-08-05 00:00:00,"Left a trail, Emitted other objects","1 of 3 i saw came close, released a small slow light, then i think they just disappeared","my girlfriend and i were on a drive and wound up on the side of the road near mt. baldey, madera canyon area and i pointed at 3 lights in the sky. curious because there were no blinkiing lights. one came very quickly towards us stoppped at a closer distance released something lit which went very slow in a circle motion then toward our direction, 

In [256]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 147890 entries, 0 to 147889
Data columns (total 13 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   Sighting          147890 non-null  int64  
 1   Occurred          147890 non-null  object 
 2   Location          147889 non-null  object 
 3   Shape             141547 non-null  object 
 4   Duration          140742 non-null  object 
 5   No of observers   141229 non-null  float64
 6   Reported          147889 non-null  object 
 7   Posted            147889 non-null  object 
 8   Characteristics   106259 non-null  object 
 9   Summary           146999 non-null  object 
 10  Text              131215 non-null  object 
 11  Location details  10205 non-null   object 
 12  Explanation       803 non-null     object 
dtypes: float64(1), int64(1), object(11)
memory usage: 14.7+ MB


In [257]:
duplicates = df.duplicated()
print(f"Number of duplicate rows: {duplicates.sum()}")

Number of duplicate rows: 0


In [258]:
missing_locations = df[df['Location'].isnull()]
missing_locations
df.loc[df['Location'].isnull(), 'Location'] = df.loc[df['Location'].isnull(), 'Summary']


In [259]:
print(df[df['Reported'].isnull()])

        Sighting                   Occurred             Location    Shape  \
136549    103020  2013-10-01 12:30:00 Local  Pittsburgh, PA, USA  Unknown   

                                     Duration  No of observers Reported  \
136549  Reported: 2013-10-07 06:22:50 Pacific              NaN      NaN   

                     Posted Characteristics  \
136549  2013-10-14 00:00:00             NaN   

                                                                                                             Summary  \
136549  I'm conflicted.  I had another sighting, mr Davenport, and I was challenged by you of filing a false report.   

                                                                                                                                                                                                                                                                                                                                                                      

In [260]:
missing_reported_dates = df[df['Reported'].isnull()]
missing_reported_dates["Reported"] = missing_reported_dates["Duration"]
missing_reported_dates["Duration"] = np.nan

df.loc[df['Reported'].isnull(), 'Reported'] = df.loc[df['Reported'].isnull(), 'Duration']
df.loc[df['Reported'].isnull(), 'Duration'] = np.nan

C:\Users\Joshu\AppData\Local\Temp\ipykernel_11124\1369677863.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  missing_reported_dates["Reported"] = missing_reported_dates["Duration"]
C:\Users\Joshu\AppData\Local\Temp\ipykernel_11124\1369677863.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  missing_reported_dates["Duration"] = np.nan


In [261]:
print(df[df['Reported'].isnull()])

Empty DataFrame
Columns: [Sighting, Occurred, Location, Shape, Duration, No of observers, Reported, Posted, Characteristics, Summary, Text, Location details, Explanation]
Index: []


In [262]:
print(df['Location'].value_counts().head(200).to_string())

Location
Phoenix, AZ, USA                             770
Seattle, WA, USA                             742
Las Vegas, NV, USA                           646
Portland, OR, USA                            589
Los Angeles, CA, USA                         550
San Diego, CA, USA                           546
Tucson, AZ, USA                              469
Chicago, IL, USA                             444
Houston, TX, USA                             429
Albuquerque, NM, USA                         398
Orlando, FL, USA                             385
Austin, TX, USA                              361
Miami, FL, USA                               354
Denver, CO, USA                              337
Boise, ID, USA                               326
Sacramento, CA, USA                          323
Myrtle Beach, SC, USA                        303
San Antonio, TX, USA                         285
San Jose, CA, USA                            285
Jacksonville, FL, USA                        274
Colorado Sp

In [ ]:

df["Location"] = (
    df["Location"]
    .str.replace(r'\([^)]*\)', '', regex=True)
    .str.lower()
    .str.replace(r'\s*,\s*', ', ', regex=True)  # FIX THIS
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
)
#Get the most common locations as reference for fuzzy matching
print(df['Location'].value_counts())
locations_common = (
    df["Location"]
    .value_counts()
    .loc[lambda x: x > 30]
    .sort_values(ascending=True)
)
print(locations_common[0:20])
# Remove rows whose Location is in locations_common
df_filtered = df[~df["Location"].isin(locations_common.index)]

#print(df_filtered.head())

Location
phoenix, az, usa                     802
new york city, ny, usa               800
seattle, wa, usa                     794
las vegas, nv, usa                   690
portland, or, usa                    607
                                    ... 
saga, , japan                          1
leamington, on, canada                 1
bullock, nc, usa                       1
marina del rey/mar vista, ca, usa      1
brandywine, de, usa                    1
Name: count, Length: 32291, dtype: int64
Location
russellville, ar, usa       31
lexington, sc, usa          31
punta gorda, fl, usa        31
pahrump, nv, usa            31
pasco, wa, usa              31
klamath falls, or, usa      31
highlands ranch, co, usa    31
sonora, ca, usa             31
upland, ca, usa             31
terrace, bc, canada         31
mcminnville, or, usa        31
midland, tx, usa            31
barstow, ca, usa            31
farmington, nm, usa         31
silverdale, wa, usa         31
mojave, ca, usa          

In [264]:
print(df_filtered["Location"].value_counts().head(20))

Location
burlington, vt, usa           30
niagara falls, on, canada     30
peterborough, on, canada      30
sunnyvale, ca, usa            30
gulf shores, al, usa          30
poughkeepsie, ny, usa         30
jackson, nj, usa              30
ormond beach, fl, usa         30
sequim, wa, usa               30
santee, ca, usa               30
rochester, nh, usa            30
centralia, wa, usa            30
levittown, ny, usa            30
alpharetta, ga, usa           30
carolina beach, nc, usa       30
crystal lake, il, usa         30
waukesha, wi, usa             30
titusville, fl, usa           30
coventry, , united kingdom    30
stanwood, wa, usa             30
Name: count, dtype: int64


In [ ]:
#Try and fix some spelling errors in the Location column using fuzzy matching with rapidfuzz

for loc in locations_common.index:

    # get best fuzzy matches
    results = rapidfuzz.process.extract(
        loc,
        df_filtered["Location"],
        scorer=rapidfuzz.fuzz.WRatio,   # better than token_sort_ratio
        limit=20
    )

    loc_parts = [p.strip() for p in loc.split(",")]
    if len(loc_parts) < 3:
        continue

    loc_city, loc_state, loc_country = loc_parts[0], loc_parts[1], loc_parts[2]

    for match, score, idx in results:

        if score < 90:
            continue

        match_parts = [p.strip() for p in match.split(",")]
        if len(match_parts) < 3:
            continue

        match_city, match_state, match_country = match_parts[0], match_parts[1], match_parts[2]

        # --- HARD FILTERS (IMPORTANT) ---
        if loc_country != match_country:
            continue

        if loc_state != match_state:
            continue

        # city similarity check (extra safety)
        city_score = rapidfuzz.fuzz.ratio(loc_city, match_city)
        if city_score < 95:
            continue

        # final update
        df.loc[idx, "Location"] = loc

        print(f"Updated '{match}' → '{loc}' (score={score}, city_score={city_score})")

Updated 'russelville, ar, usa' → 'russellville, ar, usa' (score=97.5609756097561, city_score=95.65217391304348)
Updated 'punt gorda, fl, usa' → 'punta gorda, fl, usa' (score=97.43589743589743, city_score=95.23809523809523)
Updated 'silveradale, wa, usa' → 'silverdale, wa, usa' (score=97.43589743589743, city_score=95.23809523809523)
Updated 'silverfdale, wa, usa' → 'silverdale, wa, usa' (score=97.43589743589743, city_score=95.23809523809523)
Updated 'gainsville, ga, usa' → 'gainesville, ga, usa' (score=97.43589743589743, city_score=95.23809523809523)
Updated 'sierra vita, az, usa' → 'sierra vista, az, usa' (score=97.5609756097561, city_score=95.65217391304348)
Updated 'new bedfor, ma, usa' → 'new bedford, ma, usa' (score=97.43589743589743, city_score=95.23809523809523)
Updated 'placervile, ca, usa' → 'placerville, ca, usa' (score=97.43589743589743, city_score=95.23809523809523)
Updated 'south kingston, ri, usa' → 'south kingstown, ri, usa' (score=97.87234042553192, city_score=96.5517241

In [266]:
print(df["Location"].value_counts())

Location
phoenix, az, usa          802
new york city, ny, usa    802
seattle, wa, usa          794
las vegas, nv, usa        690
portland, or, usa         607
                         ... 
wimberly, tx, usa           1
tarkio, mo, usa             1
garwood, tx, usa            1
winnebago, ne, usa          1
montague, pe, canada        1
Name: count, Length: 32214, dtype: int64


In [ ]:
#save the cleaned dataset
df.to_csv('nuforc_str_cleaned.csv', index=False)